In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed( S)
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for filename in filenames:
    print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

NameError: name 'filenames' is not defined

Installation et Imports


In [ ]:
!pip install ultralytics -q
import os
import xml.etree.ElementTree as ET
from glob import glob
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import shutil

Conversion XML vers YOLO (Format TXT)

In [ ]:
# 1. Création forcée des dossiers
os.makedirs('/kaggle/working/datasets/images', exist_ok=True)
os.makedirs('/kaggle/working/datasets/labels', exist_ok=True)

# 2. Fonction de conversion (XML -> TXT)
def convert_xml_to_yolo(xml_path, output_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    width = int(root.find('size/width').text)
    height = int(root.find('size/height').text)
    with open(output_path, 'w') as f:
        for obj in root.findall('object'):
            xmlbox = obj.find('bndbox')
            xmin, ymin = int(xmlbox.find('xmin').text), int(xmlbox.find('ymin').text)
            xmax, ymax = int(xmlbox.find('xmax').text), int(xmlbox.find('ymax').text)
            f.write(f"0 {(xmin+xmax)/(2*width)} {(ymin+ymax)/(2*height)} {(xmax-xmin)/width} {(ymax-ymin)/height}\n")

# 3. On remplit les dossiers
xml_files = glob('/kaggle/input/**/*.xml', recursive=True)
for xml_file in xml_files:
    name = os.path.basename(xml_file).replace('.xml', '')
    img_in = xml_file.replace('annotations', 'images').replace('.xml', '.png')
    if os.path.exists(img_in):
        shutil.copy(img_in, f'/kaggle/working/datasets/images/{name}.png')
        convert_xml_to_yolo(xml_file, f'/kaggle/working/datasets/labels/{name}.txt')

print(f"✅ Prêt ! {len(os.listdir('/kaggle/working/datasets/images'))} images prêtes.")

l'Entraînement

In [ ]:
from ultralytics import YOLO

# 1. On crée le fichier de configuration (le guide pour l'IA)
yaml_content = """
train: /kaggle/working/datasets/images
val: /kaggle/working/datasets/images
nc: 1
names: ['plate']
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

# 2. Charger le modèle de base (YOLOv8 nano - le plus rapide)
model = YOLO('yolov8n.pt') 

# 3. Lancer l'apprentissage (Training)
# On fait 50 époques (50 passages sur les images)
results = model.train(
    data='/kaggle/working/data.yaml', 
    epochs=50, 
    imgsz=640, 
    device=0  # Force l'utilisation du GPU
)

print("🎯 Entraînement terminé ! Ton modèle est prêt.")

test

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# 1. On charge ton modèle "champion" entraîné sur les 433 images
model_final = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# 2. Ton chemin spécifique vers la voiture orange
image_path = "/kaggle/input/datasets/ahmedbezeidrim/voiiture1/v1.jpeg" 

# 3. Lancer la détection (on met une confiance de 0.25 par sécurité)
results = model_final.predict(source=image_path, conf=0.25)

# 4. Affichage du résultat final
if len(results[0].boxes) > 0:
    for r in results:
        im_array = r.plot()  # Dessine le rectangle et 'plate'
        plt.figure(figsize=(12, 10))
        plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("Plaque détectée par ton IA !")
        plt.show()
    print(f"✅ Succès ! {len(results[0].boxes)} plaque(s) trouvée(s).")
else:
    print("❌ L'IA n'a pas trouvé la plaque. Essaie de baisser 'conf=0.10' dans le code.")

# 5. Bonus : Afficher les coordonnées exactes
for box in results[0].boxes:
    print(f"📍 Coordonnées du rectangle : {box.xyxy[0].tolist()}")

In [ ]:
!pip install easyocr -q

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt
import numpy as np

# 1. Initialiser le lecteur de texte (en mode anglais/chiffres latins)
reader = easyocr.Reader(['en']) 

# 2. Charger ton modèle YOLO
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# 3. Ton image mauritanienne
image_path = "/kaggle/input/datasets/ahmedbezeidrim/voiture/OIP.jpg"
img = cv2.imread(image_path)

# 4. YOLO trouve la plaque
results = model.predict(source=image_path, conf=0.25)

for r in results:
    for box in r.boxes:
        # 1. On récupère les coordonnées
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 2. On découpe avec une marge de 10% pour ne pas couper les lettres
        h, w = y2 - y1, x2 - x1
        plate_img = img[max(0, y1-int(h*0.1)):y2+int(h*0.1), 
                        max(0, x1-int(w*0.1)):x2+int(w*0.1)]

        # 3. SUPER-RESOLUTION MANUELLE : Agrandir l'image (Interpolation Lanczos)
        # On multiplie la taille par 3 pour que l'OCR voie de "grosses" lettres
        plate_img = cv2.resize(plate_img, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)

        # 4. NETTOYAGE DU FLOU (Denoising)
        # On enlève le "grain" de l'image
        plate_img = cv2.fastNlMeansDenoisingColored(plate_img, None, 10, 10, 7, 21)

        # 5. ACCENTUATION (Sharpening)
        # On crée un filtre pour rendre les bords des lettres très "tranchants"
        kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
        plate_img = cv2.filter2D(plate_img, -1, kernel)

       # 6. PASSAGE EN GRIS + NETTOYAGE MORPHOLOGIQUE
gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
# On utilise un seuillage adaptatif (plus doux que le précédent)
thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

# On affine les lettres pour les décoller
kernel = np.ones((2,2), np.uint8)
# Erosion : ça réduit un peu l'épaisseur pour séparer le WW
cleaned = cv2.erode(thresh, kernel, iterations=1) 
# On remet en noir sur blanc pour EasyOCR
final_for_ocr = cv2.bitwise_not(cleaned)

plt.imshow(final_for_ocr, cmap='gray')
plt.show()

# Lecture
text = reader.readtext(final_for_ocr, detail=0)
print(f"🎯 Résultat final : {' '.join(text)}")